In [ ]:
import os
import time

import chromadb
import numpy as np
import pandas as pd
from cohere import ClientV2
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# also includes intro_to_vector_db_using_chroma_db

In [2]:
### Load the metadata
df = pd.read_csv('arxiv_papers_5k.csv')
print(f"Loaded {len(df)} papers")

### Load the embeddings
embeddings = np.load('embeddings_cohere_5k.npy')
print(f"Loaded embeddings with shape: {embeddings.shape}")
print(f"Each paper is represented by a {embeddings.shape[1]}-dimensional vector")

### Verify they match
assert len(df) == len(embeddings), "Mismatch between papers and embeddings!"

### Check the distribution across categories
print(f"\nPapers per category:")
print(df['category'].value_counts().sort_index())

### Look at a sample paper
print(f"\nSample paper:")
print(f"Title: {df['title'].iloc[0]}")
print(f"Category: {df['category'].iloc[0]}")
print(f"Abstract: {df['abstract'].iloc[0][:200]}...")

Loaded 5000 papers
Loaded embeddings with shape: (5000, 1536)
Each paper is represented by a 1536-dimensional vector

Papers per category:
category
cs.CL    1000
cs.CV    1000
cs.DB    1000
cs.LG    1000
cs.SE    1000
Name: count, dtype: int64

Sample paper:
Title: Optimizing Mixture of Block Attention
Category: cs.LG
Abstract: Mixture of Block Attention (MoBA) (Lu et al., 2025) is a promising building block for efficiently processing long contexts in LLMs by enabling queries to sparsely attend to a small subset of key-value...


In [4]:
### Initialize ChromaDB in-memory client (data only exists while script runs)
client = chromadb.Client()

### Create a collection
collection = client.create_collection(
    name="arxiv_papers",
    metadata={"description": "5000 arXiv papers from computer science"}
)

print(f"Created collection: {collection.name}")
print(f"Collection count: {collection.count()}")

Created collection: arxiv_papers
Collection count: 0


In [5]:
### Prepare the data for ChromaDB
### ChromaDB wants: IDs, embeddings, metadata, and optional documents
ids = [f"paper_{i}" for i in range(len(df))]
metadatas = [
    {
        "title": row['title'],
        "category": row['category'],
        "year": int(str(row['published'])[:4]),  # Store year as integer for filtering
        "authors": row['authors'][:100] if len(row['authors']) <= 100 else row['authors'][:97] + "..."
    }
    for _, row in df.iterrows()
]
documents = df['abstract'].tolist()

In [6]:
### Insert in batches to respect the ~5,461 embedding limit
batch_size = 5000  # Safe batch size well under the limit
print(f"Inserting {len(embeddings)} embeddings in batches of {batch_size}...")

for i in range(0, len(embeddings), batch_size):
    batch_end = min(i + batch_size, len(embeddings))
    print(f"  Batch {i//batch_size + 1}: Adding papers {i} to {batch_end}")

    collection.add(
        ids=ids[i:batch_end],
        embeddings=embeddings[i:batch_end].tolist(),
        metadatas=metadatas[i:batch_end],
        documents=documents[i:batch_end]
    )

print(f"\nCollection now contains {collection.count()} papers")

Inserting 5000 embeddings in batches of 5000...
  Batch 1: Adding papers 0 to 5000

Collection now contains 5000 papers


In [8]:
load_dotenv()
cohere_api_key = os.getenv('COHERE_API_KEY')

if not cohere_api_key:
    raise ValueError(
        "COHERE_API_KEY not found. Make sure you have a .env file with your API key."
    )

co = ClientV2(api_key=cohere_api_key)
print("✓ Cohere API key loaded")

✓ Cohere API key loaded


In [9]:
### First, embed the query using Cohere (same model as our documents)
query_text = "neural network training and optimization techniques"

response = co.embed(
    texts=[query_text],
    model='embed-v4.0',
    input_type='search_query',
    embedding_types=['float']
)
query_embedding = np.array(response.embeddings.float_[0])

print(f"Query: '{query_text}'")
print(f"Query embedding shape: {query_embedding.shape}")

### Now search the collection
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

### Display the results
print(f"\nTop 5 most similar papers:")
print("=" * 80)

for i in range(len(results['ids'][0])):
    paper_id = results['ids'][0][i]
    distance = results['distances'][0][i]
    metadata = results['metadatas'][0][i]

    print(f"\n{i+1}. {metadata['title']}")
    print(f"   Category: {metadata['category']} | Year: {metadata['year']}")
    print(f"   Distance: {distance:.4f}")
    print(f"   Abstract: {results['documents'][0][i][:150]}...")

Query: 'neural network training and optimization techniques'
Query embedding shape: (1536,)

Top 5 most similar papers:

1. Training Neural Networks at Any Scale
   Category: cs.LG | Year: 2025
   Distance: 1.1139
   Abstract: This article reviews modern optimization methods for training neural networks with an emphasis on efficiency and scale. We present state-of-the-art op...

2. On the Convergence of Overparameterized Problems: Inherent Properties of the Compositional Structure of Neural Networks
   Category: cs.LG | Year: 2025
   Distance: 1.2552
   Abstract: This paper investigates how the compositional structure of neural networks shapes their optimization landscape and training dynamics. We analyze the g...

3. A Distributed Training Architecture For Combinatorial Optimization
   Category: cs.LG | Year: 2025
   Distance: 1.3013
   Abstract: In recent years, graph neural networks (GNNs) have been widely applied in tackling combinatorial optimization problems. However, existing me

In [ ]:
"""
ChromaDB uses squared L2 distance by default.
For normalized embeddings (like Cohere's),
there's a mathematical relationship: distance ≈ 2(1 - cosine_similarity).
So a distance of 1.16 corresponds to a cosine similarity of about 0.42.
"""

In [11]:

def numpy_search(query_embedding, embeddings, top_k=5):
    """Brute-force similarity search using NumPy"""
    # Calculate cosine similarity between query and all papers
    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        embeddings
    )[0]

    # Get top k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]

    return top_indices

### Generate a query embedding (using one of our paper embeddings as a proxy)
query_embedding = embeddings[0]

### Test NumPy approach
start_time = time.time()
for _ in range(100):  # Run 100 queries to get stable timing
    top_indices = numpy_search(query_embedding, embeddings, top_k=5)
numpy_time = (time.time() - start_time) / 100 * 1000  # Convert to milliseconds

print(f"NumPy brute-force search (5000 papers): {numpy_time:.2f} ms per query")

NumPy brute-force search (5000 papers): 22.39 ms per query


In [12]:
### Test ChromaDB approach (query using the embedding directly)
start_time = time.time()
for _ in range(100):
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=5
    )
chromadb_time = (time.time() - start_time) / 100 * 1000

print(f"ChromaDB search (5000 papers): {chromadb_time:.2f} ms per query")
print(f"\nSpeedup: {numpy_time / chromadb_time:.1f}x faster")

ChromaDB search (5000 papers): 1.31 ms per query

Speedup: 17.1x faster


In [ ]:
"""
HNSW is an approximate algorithm.
It doesn't guarantee finding the absolute closest papers,
but it finds very close papers very quickly. This trade-off is controlled by parameters:

ef_construction: How carefully the index is built (higher = better quality, slower build)
ef_search: How thoroughly queries search (higher = better recall, slower queries)
M: Number of connections per point (higher = better search, more memory)
"""

In [ ]:
### Compare ChromaDB results to exact NumPy results
query_embedding = embeddings[100]

### Get top 10 from NumPy (exact)
numpy_results = numpy_search(query_embedding, embeddings, top_k=10)

### Get top 10 from ChromaDB (approximate)
chromadb_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

### Extract paper indices from ChromaDB results (convert "paper_123" to 123)
chromadb_indices = [int(id.split('_')[1]) for id in chromadb_results['ids'][0]]

### Calculate overlap
overlap = len(set(numpy_results) & set(chromadb_indices))

print(f"NumPy top 10 (exact): {numpy_results}")
print(f"ChromaDB top 10 (approximate): {chromadb_indices}")
print(f"\nOverlap: {overlap}/10 papers match")
print(f"Recall@10: {overlap/10*100:.1f}%")

In [13]:
### Compare ChromaDB results to exact NumPy results
query_embedding = embeddings[100]

### Get top 10 from NumPy (exact)
numpy_results = numpy_search(query_embedding, embeddings, top_k=10)

### Get top 10 from ChromaDB (approximate)
chromadb_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

### Extract paper indices from ChromaDB results (convert "paper_123" to 123)
chromadb_indices = [int(id.split('_')[1]) for id in chromadb_results['ids'][0]]

### Calculate overlap
overlap = len(set(numpy_results) & set(chromadb_indices))

print(f"NumPy top 10 (exact): {numpy_results}")
print(f"ChromaDB top 10 (approximate): {chromadb_indices}")
print(f"\nOverlap: {overlap}/10 papers match")
print(f"Recall@10: {overlap/10*100:.1f}%")

NumPy top 10 (exact): [ 100  984  509 2261 3044  701 1055  830 3410 1311]
ChromaDB top 10 (approximate): [100, 984, 509, 2261, 3044, 701, 1055, 830, 3410, 1311]

Overlap: 10/10 papers match
Recall@10: 100.0%


In [ ]:
"""
For 5,000 papers with 1536-dimensional embeddings, we're looking at roughly 90-100MB of RAM.
This scales linearly: 10,000 papers would be about 180-200MB, 50,000 papers about 900MB-1GB.
"""

In [ ]:
# ### Persistent storage (data saved to disk)
# client = chromadb.PersistentClient(path="./chroma_db")

### ChromaDB's HNSW index grows but never shrinks
"""
If 5,000 documents are added then 4,000 deleted, the index still uses memory for 5,000.
The only way to reclaim this space is to create a new collection and re-add the documents we want to keep.

It's a known limitation with HNSW indexes.
Not a bug, but a fundamental trade-off for the algorithm's speed.
Keep in mind when designing systems that frequently add and remove documents.
"""

### Default embedding function
"""
on calling collection.query(query_texts=["some text"]),
ChromaDB automatically embeds your query using its default model (all-MiniLM-L6-v2).
"""

In [ ]:
"""
ChromaDB is perfect for:

- Learning and prototyping: Get immediate feedback without infrastructure setup
- Local development: No internet required, no API costs
- Small to medium datasets: Up to 100,000 documents on a standard laptop
- Single-machine applications: Desktop tools, local RAG systems, personal assistants
- Rapid experimentation: Test different embedding models or chunking strategies


Move to production databases when you need:

- Massive scale: Millions of vectors or high query volume (thousands of QPS)
- Distributed deployment: Multiple machines, load balancing, high availability
- Advanced features: Hybrid search, multi-tenancy, access control, backup/restore
- Production SLAs: Guaranteed uptime, support, monitoring
- Team collaboration: Multiple developers working with shared data
"""

In [ ]:
### Helper function to make querying easier
def search_papers(query_text, n_results=5):
    """Search papers using semantic similarity"""
    # Embed the query
    response = co.embed(
        texts=[query_text],
        model='embed-v4.0',
        input_type='search_query',
        embedding_types=['float']
    )
    query_embedding = np.array(response.embeddings.float_[0])

    # Search
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    return results

In [ ]:
### 1. Find papers about a specific topic
results = search_papers("reinforcement learning and robotics")

### 2. Try a different domain
results_cv = search_papers("image segmentation techniques")

### 3. Test with a broad query
results_broad = search_papers("deep learning applications")

In [ ]:
"""
Skills learnt;

- Loading pre-generated embeddings and metadata
- Creating and querying ChromaDB collections
- Running pure vector similarity searches
- Comparing approximate vs exact search quality
- Understanding when to use ChromaDB vs production databases
"""

'\nSkills learn;\n'